In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch


# ==========================================================
# MODEL PATH
# ==========================================================

CHECKPOINT = "./V5C_Final_Merged_Model"


# ==========================================================
# LOAD MODEL
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 100)
print("LOADING V5C MODEL")
print("=" * 100)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT,
    trust_remote_code=True,
)

print("Loading merged model...")

model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

print("✅ V5C Model Loaded Successfully")



g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


LOADING V5C MODEL
Loading tokenizer...
Loading merged model...


Loading checkpoint shards: 100%|██████████| 5/5 [01:04<00:00, 12.91s/it]

✅ V5C Model Loaded Successfully


In [5]:

# ==========================================================
# SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """You are Compatifi V5C.

Your ONLY task is to generate a concise factual summary
of the current conversation.

V5C summarizes WHAT HAPPENED in the conversation.

You must output EXACTLY one JSON object containing
EXACTLY one key:

{
  "summary": ""
}

FIELD DEFINITION:

summary:
A concise factual summary of the conversation in 1–2 sentences.

RULES:

1. Summarize the conversation.
2. Focus on the main topic.
3. Include important facts.
4. Include the user's situation or goal when relevant.
5. Include the key outcome or issue when relevant.
6. Use only information supported by the conversation.
7. Do NOT invent information.
8. Do NOT generate a reply.
9. Do NOT give advice.
10. Do NOT provide recommendations.
11. Do NOT extract long-term memories.
12. Do NOT build a user profile.
13. Do NOT build an other-person profile.
14. Do NOT analyze compatibility.
15. Do NOT analyze personality.
16. Do NOT include unnecessary details.
17. Very little information must still produce a short factual summary.
18. Never return an empty summary when factual information is available.
19. Keep the summary concise.
20. Prefer 1–2 sentences.
21. Return JSON only.
22. Do not output markdown.
23. Do not output <think>.
24. Do not add any fields other than "summary".
25. Your output must begin with { and end with }.

The required output format is:

{
  "summary": ""
}
"""


# ==========================================================
# TEST CONVERSATIONS
# ==========================================================

# test_conversations = [

#     {
#         "name": "V5C TEST 1 - Complete / Rich",
#         "relationship": "Manager",
#         "conversation": """
# Manager: How is the new project going?
# User: The development is mostly finished, but we're still fixing performance issues.
# Manager: Are you still on track for Friday?
# User: Yes, I should finish the optimization by Thursday and deploy Friday morning.
# Manager: Good. Please send me the final performance report before deployment.
# User: Sure, I'll send it Thursday afternoon.
# """
#     },

#     {
#         "name": "V5C TEST 2 - Partial Information",
#         "relationship": "Friend",
#         "conversation": """
# Friend: How did your interview go?
# User: It went okay. Some questions were difficult, but I think I handled them fairly well.
# Friend: That's good to hear.
# """
#     },

#     {
#         "name": "V5C TEST 3 - Very Little Information",
#         "relationship": "Friend",
#         "conversation": """
# Friend: Hey, how are you?
# User: I'm good.
# Friend: Great.
# """
#     },

#     {
#         "name": "V5C TEST 4 - Problem / Work",
#         "relationship": "Colleague",
#         "conversation": """
# Colleague: Did you finish the deployment?
# User: Not yet. The API container keeps failing during startup.
# Colleague: Did you check the logs?
# User: Yes. The environment variable isn't being loaded correctly.
# Colleague: So you're going to check the deployment configuration?
# User: Yes, I'll compare it with the previous working version.
# """
#     },

#     {
#         "name": "V5C TEST 5 - Emotional Situation",
#         "relationship": "Partner",
#         "conversation": """
# Partner: You seem stressed lately.
# User: Work has been overwhelming and I'm not sure where to begin.
# Partner: You've had a lot going on recently.
# User: Yeah. I feel like everything is piling up at once.
# """
#     },

#     {
#         "name": "V5C TEST 6 - Goal / Career",
#         "relationship": "Friend",
#         "conversation": """
# Friend: Are you still learning Python?
# User: Yes. I practice every evening because I want to become a backend developer.
# Friend: How is it going?
# User: I'm getting better with APIs and databases, but I still need more practice.
# """
#     },

#     {
#         "name": "V5C TEST 7 - Decision",
#         "relationship": "Friend",
#         "conversation": """
# Friend: Which laptop are you thinking about buying?
# User: I need one for programming and college.
# Friend: What's your budget?
# User: Around $1,200. I want something reliable that will last several years.
# """
#     },

#     {
#         "name": "V5C TEST 8 - Success / Outcome",
#         "relationship": "Friend",
#         "conversation": """
# Friend: Did you hear back from the company?
# User: Yes! I got the job!
# Friend: That's amazing!
# User: I can't believe it. I've been waiting for this opportunity for months.
# """
#     },

# ]



test_conversations = [

    # ==========================================================
    # USER-PRIMARY
    # ==========================================================

    {
        "name": "V5C TEST 1 - Complete / Rich (User Primary)",
        "relationship": "Manager",
        "conversation": """
Manager: How is the new project going?
User: The development is mostly finished, but we're still fixing performance issues.
Manager: Are you still on track for Friday?
User: Yes, I should finish the optimization by Thursday and deploy Friday morning.
Manager: Good. Please send me the final performance report before deployment.
User: Sure, I'll send it Thursday afternoon.
"""
    },

    {
        "name": "V5C TEST 2 - Partial Information (User Primary)",
        "relationship": "Friend",
        "conversation": """
Friend: How did your interview go?
User: It went okay. Some questions were difficult, but I think I handled them fairly well.
Friend: That's good to hear.
"""
    },

    # ==========================================================
    # OTHER-PERSON PRIMARY
    # Tests whether V5C avoids automatically prioritizing User
    # ==========================================================

    {
        "name": "V5C TEST 3 - Other Person Primary / Emotional",
        "relationship": "Friend",
        "conversation": """
User: You seem quiet today. Is everything okay?
Friend: Honestly, I'm worried about my exams next week.
User: Have you had enough time to prepare?
Friend: Not really. I've been struggling to focus and I'm afraid I won't do well.
User: I'm sorry you're feeling that way. We can study together if you want.
"""
    },

    {
        "name": "V5C TEST 4 - Other Person Primary / Career",
        "relationship": "Partner",
        "conversation": """
User: How was your meeting with your manager?
Partner: It went better than I expected. They offered me a leadership role on the new project.
User: That's amazing. Are you going to accept it?
Partner: I think so, but I'm nervous about taking on more responsibility.
User: You'll do great. You've been working toward something like this for a long time.
"""
    },

    {
        "name": "V5C TEST 5 - Other Person Primary / Problem",
        "relationship": "Sibling",
        "conversation": """
User: You haven't been answering your phone much lately.
Sibling: I've been dealing with some financial problems and didn't want to worry anyone.
User: I'm sorry. Is there anything I can do?
Sibling: I don't know yet. I'm trying to figure out a plan first.
"""
    },

    # ==========================================================
    # BOTH PARTICIPANTS PRIMARY
    # Tests whether V5C preserves shared plans/decisions/conflicts
    # ==========================================================

    {
        "name": "V5C TEST 6 - Both Primary / Shared Plan",
        "relationship": "Friend",
        "conversation": """
User: We should finally plan that hiking trip.
Friend: Definitely. I'm free the second weekend of next month.
User: That works for me too.
Friend: Great. Let's decide on the trail after checking the weather forecast.
User: Sounds good. I'll research a few options.
"""
    },

    {
        "name": "V5C TEST 7 - Both Primary / Conflict Resolution",
        "relationship": "Partner",
        "conversation": """
User: I feel like we've both been too busy to spend much time together.
Partner: I know. I've been feeling the same way.
User: I don't want us to keep putting it off.
Partner: Neither do I. Let's keep Saturday evenings free for each other.
User: I like that idea. Let's try it starting this week.
"""
    },

    {
        "name": "V5C TEST 8 - Both Primary / Joint Decision",
        "relationship": "Colleague",
        "conversation": """
User: Should we delay the release until the performance issue is fixed?
Colleague: I think that's safer than deploying something unstable.
User: Agreed. We can spend another two days testing.
Colleague: Let's move the release to Wednesday and inform the team.
User: Okay, I'll update everyone today.
"""
    },

    # ==========================================================
    # NEUTRAL EVENT PRIMARY
    # Tests event-centric summaries
    # ==========================================================

    {
        "name": "V5C TEST 9 - Neutral Event / Schedule Change",
        "relationship": "Team Member",
        "conversation": """
Team Member: Did you see the announcement about the conference?
User: No, what happened?
Team Member: It was postponed because the venue had an unexpected scheduling conflict.
User: Do they know the new date yet?
Team Member: Not yet. They'll announce it next week.
"""
    },

    {
        "name": "V5C TEST 10 - Neutral Event / External Problem",
        "relationship": "Friend",
        "conversation": """
Friend: Our flight was canceled.
User: Seriously? Did the airline say why?
Friend: Bad weather caused delays across the airport.
User: Are they putting us on another flight?
Friend: Yes, but the earliest available one is tomorrow morning.
"""
    },

    # ==========================================================
    # VERY LITTLE INFORMATION
    # ==========================================================

    {
        "name": "V5C TEST 11 - Very Little Information",
        "relationship": "Friend",
        "conversation": """
Friend: Hey, how are you?
User: I'm good.
Friend: Great.
"""
    },

    # ==========================================================
    # ATTRIBUTION TEST
    # Important: facts must remain assigned to correct person
    # ==========================================================

    {
        "name": "V5C TEST 12 - Attribution Accuracy",
        "relationship": "Partner",
        "conversation": """
User: I can't believe you kept this a secret.
Partner: I've been saving money for months.
User: For what?
Partner: I wanted to surprise you with a trip to Italy for our anniversary.
User: Italy has always been my dream destination.
Partner: I know. That's why I wanted to make it special.
"""
    },

]



























# ==========================================================
# RUN TESTS
# ==========================================================

for test in test_conversations:

    print("\n")
    print("=" * 100)
    print(test["name"])
    print("=" * 100)

    # ------------------------------------------------------
    # USER PROMPT
    # ------------------------------------------------------

    user_prompt = f"""
Relationship: {test["relationship"]}

Conversation:

{test["conversation"]}

Instruction:
Generate a concise factual summary of the conversation.
"""

    # ------------------------------------------------------
    # CHAT MESSAGES
    # ------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    # ------------------------------------------------------
    # CHAT TEMPLATE
    # ------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # ------------------------------------------------------
    # TOKENIZE
    # ------------------------------------------------------

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    # ------------------------------------------------------
    # GENERATE
    # ------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
        )

    # ------------------------------------------------------
    # DECODE ONLY NEW TOKENS
    # ------------------------------------------------------

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    # ------------------------------------------------------
    # PRINT RESULT
    # ------------------------------------------------------

    print("\nMODEL OUTPUT")
    print("-" * 100)
    print(response)

    print("\n")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.




V5C TEST 1 - Complete / Rich (User Primary)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

The user updated their manager on the project, confirming that development is mostly complete and deployment is scheduled for Friday morning. The user agreed to send the final performance report by Thursday afternoon.




V5C TEST 2 - Partial Information (User Primary)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...


MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

The team member informed the user that the conference was postponed due to a venue conflict, with a new date to be announced next week.




V5C TEST 10 - Neutral Event / External Problem


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

The friends discussed their canceled flight due to weather, and the airline rescheduled them for the following morning.




V5C TEST 11 - Very Little Information


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

The user and their friend briefly exchanged casual greetings and confirmed they are both doing well.




V5C TEST 12 - Attribution Accuracy

MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

The partner surprised the user by revealing they saved money to fund an anniversary trip to Italy. The user expressed excitement about the destination.


